# Model Training and Validation

Models Used:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

Each model will be trained using the same training dataset.

The models will be compared based on:

- Training Accuracy
- Validation Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

Training and validation performance will also be compared to identify possible overfitting or underfitting.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)

In [2]:
train_df = pd.read_csv("../outputs/train_data.csv")

print("Training dataset loaded.")
print(train_df.shape)

Training dataset loaded.
(9764, 22)


In [3]:
X = train_df.drop("Revenue", axis=1)

y = train_df["Revenue"]

print(f"Features Shape: {X.shape}")
print(f"Target Shape: {y.shape}")

Features Shape: (9764, 21)
Target Shape: (9764,)


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training and Validation Split Completed")

print(f"\nTraining Data: {X_train.shape}")
print(f"Validation Data: {X_val.shape}")

Training and Validation Split Completed

Training Data: (7811, 21)
Validation Data: (1953, 21)


In [5]:
preprocessor = joblib.load(
    "../outputs/preprocessor.pkl"
)

print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


In [6]:
models = {
    
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=42
    )
}

model_info = pd.DataFrame({
    "Model": list(models.keys()),
    
    "Algorithm Type": [
        "Linear Classification",
        "Tree-Based Classification",
        "Bagging Ensemble",
        "Boosting Ensemble"
    ],
    
    "Purpose": [
        "Baseline Model",
        "Learn Decision Rules",
        "Reduce Variance and Improve Generalization",
        "Sequentially Improve Model Performance"
    ]
})

model_info

,Model,Algorithm Type,Purpose
0,Logistic Regression,Linear Classification,Baseline Model
1,Decision Tree,Tree-Based Classification,Learn Decision Rules
2,Random Forest,Bagging Ensemble,Reduce Variance and Improve Generalization
3,Gradient Boosting,Boosting Ensemble,Sequentially Improve Model Performance


In [7]:
results = []

trained_models = {}

for model_name, model in models.items():
    
    print("=" * 60)
    print(f"Training: {model_name}")
    print("=" * 60)
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    trained_models[model_name] = pipeline
    
    train_predictions = pipeline.predict(X_train)
    val_predictions = pipeline.predict(X_val)
    
    train_probabilities = pipeline.predict_proba(X_train)[:, 1]
    val_probabilities = pipeline.predict_proba(X_val)[:, 1]
    
    train_accuracy = accuracy_score(
        y_train,
        train_predictions
    )
    
    val_accuracy = accuracy_score(
        y_val,
        val_predictions
    )
    
    precision = precision_score(
        y_val,
        val_predictions,
        zero_division=0
    )
    
    recall = recall_score(
        y_val,
        val_predictions,
        zero_division=0
    )
    
    f1 = f1_score(
        y_val,
        val_predictions,
        zero_division=0
    )
    
    roc_auc = roc_auc_score(
        y_val,
        val_probabilities
    )
    
    difference = train_accuracy - val_accuracy
    
    if difference > 0.10:
        status = "Possible Overfitting"
    
    elif train_accuracy < 0.70 and val_accuracy < 0.70:
        status = "Possible Underfitting"
    
    else:
        status = "Good Generalization"
    
    results.append({
        "Model": model_name,
        "Train Accuracy": train_accuracy,
        "Validation Accuracy": val_accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "Train-Val Difference": difference,
        "Model Status": status
    })
    
    print(f"\nValidation Accuracy: {val_accuracy:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Status: {status}")

Training: Logistic Regression

Validation Accuracy: 0.8771
ROC-AUC: 0.8714
Status: Good Generalization
Training: Decision Tree

Validation Accuracy: 0.8848
ROC-AUC: 0.8280
Status: Good Generalization
Training: Random Forest

Validation Accuracy: 0.8996
ROC-AUC: 0.9169
Status: Good Generalization
Training: Gradient Boosting

Validation Accuracy: 0.8935
ROC-AUC: 0.9152
Status: Good Generalization


In [8]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Train Accuracy,Validation Accuracy,Precision,Recall,F1 Score,ROC-AUC,Train-Val Difference,Model Status
0,Random Forest,0.959672,0.899642,0.740088,0.550820,0.631579,0.916925,0.060031,Good Generalization
1,Gradient Boosting,0.921009,0.893497,0.680297,0.600000,0.637631,0.915183,0.027512,Good Generalization
2,Logistic Regression,0.885290,0.877112,0.694611,0.380328,0.491525,0.871383,0.008178,Good Generalization
3,Decision Tree,0.930227,0.884793,0.658730,0.544262,0.596050,0.827964,0.045434,Good Generalization


In [9]:
display_results = results_df.copy()

score_columns = [
    "Train Accuracy",
    "Validation Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "Train-Val Difference"
]

for column in score_columns:
    display_results[column] = (
        display_results[column] * 100
    ).round(2)

display_results

,Model,Train Accuracy,Validation Accuracy,Precision,Recall,F1 Score,ROC-AUC,Train-Val Difference,Model Status
0,Random Forest,95.97,89.96,74.01,55.08,63.16,91.69,6.00,Good Generalization
1,Gradient Boosting,92.10,89.35,68.03,60.00,63.76,91.52,2.75,Good Generalization
2,Logistic Regression,88.53,87.71,69.46,38.03,49.15,87.14,0.82,Good Generalization
3,Decision Tree,93.02,88.48,65.87,54.43,59.61,82.80,4.54,Good Generalization


In [10]:
overfitting_analysis = results_df[
    [
        "Model",
        "Train Accuracy",
        "Validation Accuracy",
        "Train-Val Difference",
        "Model Status"
    ]
].copy()

for column in [
    "Train Accuracy",
    "Validation Accuracy",
    "Train-Val Difference"
]:
    
    overfitting_analysis[column] = (
        overfitting_analysis[column] * 100
    ).round(2)

overfitting_analysis

,Model,Train Accuracy,Validation Accuracy,Train-Val Difference,Model Status
0,Random Forest,95.97,89.96,6.00,Good Generalization
1,Gradient Boosting,92.10,89.35,2.75,Good Generalization
2,Logistic Regression,88.53,87.71,0.82,Good Generalization
3,Decision Tree,93.02,88.48,4.54,Good Generalization


In [11]:
print("MODEL GENERALIZATION ANALYSIS")
print("=" * 70)

for _, row in results_df.iterrows():
    
    print(f"\nModel: {row['Model']}")
    
    print(
        f"Training Accuracy: "
        f"{row['Train Accuracy']:.2%}"
    )
    
    print(
        f"Validation Accuracy: "
        f"{row['Validation Accuracy']:.2%}"
    )
    
    print(
        f"Difference: "
        f"{row['Train-Val Difference']:.2%}"
    )
    
    print(
        f"Status: "
        f"{row['Model Status']}"
    )

MODEL GENERALIZATION ANALYSIS

Model: Random Forest
Training Accuracy: 95.97%
Validation Accuracy: 89.96%
Difference: 6.00%
Status: Good Generalization

Model: Gradient Boosting
Training Accuracy: 92.10%
Validation Accuracy: 89.35%
Difference: 2.75%
Status: Good Generalization

Model: Logistic Regression
Training Accuracy: 88.53%
Validation Accuracy: 87.71%
Difference: 0.82%
Status: Good Generalization

Model: Decision Tree
Training Accuracy: 93.02%
Validation Accuracy: 88.48%
Difference: 4.54%
Status: Good Generalization


In [12]:
results_df.to_csv(
    "../outputs/initial_model_results.csv",
    index=False
)

print("Initial model results saved successfully.")

Initial model results saved successfully.


In [13]:
for model_name, model_pipeline in trained_models.items():
    
    file_name = (
        model_name
        .lower()
        .replace(" ", "_")
    )
    
    joblib.dump(
        model_pipeline,
        f"../outputs/{file_name}_model.pkl"
    )

print("All trained models saved successfully.")

All trained models saved successfully.


In [14]:
best_initial_model = results_df.iloc[0]

print("=" * 60)
print("NOTEBOOK 03 COMPLETED")
print("=" * 60)

print("\nModels Trained:")
for model in models.keys():
    print(f"- {model}")

print("\nCurrent Best Model Based on Validation ROC-AUC:")

print(best_initial_model["Model"])

print(
    f"ROC-AUC: "
    f"{best_initial_model['ROC-AUC']:.4f}"
)

print("\nNext Step:")
print("Cross Validation and Hyperparameter Tuning")

NOTEBOOK 03 COMPLETED

Models Trained:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

Current Best Model Based on Validation ROC-AUC:
Random Forest
ROC-AUC: 0.9169

Next Step:
Cross Validation and Hyperparameter Tuning
